In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv
import os

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
load_dotenv()

llm = ChatMistralAI(
    model="open-mistral-7b",
    api_key=os.getenv("MISTRAL_API_KEY")
)
def chat_node(state: ChatState):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)

    # response store state
    return {'messages': [response]}

In [ ]:
graph = StateGraph(ChatState)
graph.add_node("chat", chat_node)

graph.add_edge(START, "chat")
graph.add_edge("chat", END)

chatbot=graph.compile()
chatbot

In [ ]:
initial_state = {
    'messages': []
}

In [ ]:
while True:
    user_input = input("User: ")
    print("User input:", user_input)
    
    if (user_input.lower() in ['exit', 'quit']):
        print("Exiting the chatbot.")
        break
    
    else:
        # Append user input to the messages
        initial_state['messages'].append(HumanMessage(content=user_input))
    
        # Invoke the chatbot with the updated state
        response = chatbot.invoke(initial_state)
        
        # Print the latest response from the chatbot
        print("Chatbot:", response['messages'][-1].content)

In [ ]:
from langchain_core.messages import HumanMessage

state = {"messages": []}

while True:
    user_input = input("User: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Exiting chatbot...")
        break

    response = chatbot.invoke({
        "messages": [HumanMessage(content=user_input)]
    })

    print("Chatbot:", response["messages"][-1].content)